In [ ]:
%load_ext autoreload
%autoreload 2

import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

print("CUDA доступна:", torch.cuda.is_available())


## Tiktoken Tokenizer

Пример токенизации с `tiktoken`. Здесь используется `cl100k_base` — это **не** токенизатор модели Qwen, которую мы загрузим далее.


In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
texts = ["токенизатор", "Лучшая еда - это шавуха"]

for s in texts:
    ids = enc.encode(s)
    token_bytes = [enc.decode_single_token_bytes(i) for i in ids]
    print(f"\n=== {s!r} ===")
    print("token IDs:", ids)
    print("token bytes:", token_bytes)  # отдельные токены могут быть частями UTF-8 символа
    print("decoded:", enc.decode(ids))


In [ ]:
tokenizer_map = enc._mergeable_ranks  # приватный атрибут, только для изучения BPE
special_tokens = enc._special_tokens   

In [ ]:
special_tokens

In [ ]:
len(tokenizer_map)

In [ ]:
tokenizer_map

In [ ]:
key = 'уч'.encode('utf-8')
key

In [ ]:
tokenizer_map[key]

## Загрузка модели с Hugging Face локально

В Colab выберите среду выполнения с GPU. Для учебного примера используем сравнительно небольшую Qwen2.5-1.5B-Instruct; dtype FP16 уменьшает потребление GPU-памяти по сравнению с FP32.


In [ ]:
# Загружаем модель и соответствующий ей токенизатор
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

model_hf = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
model_hf

In [ ]:
# конфиг модели
model_hf.config

In [ ]:
tokenizer

In [ ]:
def invoke(model, tokenizer, prompt, system_prompt=None, config=None):
    device = model.device

    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    # Применяем шаблон чата: роли, спецтокены и маркер начала ответа
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    print(f"Input as text:\n---\n{text}", end="---\n\n")

    # Преобразуем текст в ID токенов
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    print(f"Input as token ids:\n---\n{model_inputs.input_ids}", end="\n---\n\n")

    kwargs = {"max_new_tokens": 256} if config is None else {"generation_config": config}
    with torch.inference_mode():
        generated_ids = model.generate(**model_inputs, **kwargs)

    # Отделяем новые токены от исходного промпта
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    print(f"Output as token ids:\n---\n{generated_ids}", end="\n---\n\n")

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"Output as text:\n---\n{response}", end="\n---\n")
    return response


In [ ]:
system_prompt = 'You are math assistant. Solve the provided math problem step by step and put your final answer within \\boxed{}'
prompt = 'Реши квадратное уравнение: x**2 + 1 = 0'

output = invoke(model_hf, tokenizer, prompt=prompt, system_prompt=system_prompt,)

### Поговорим немного о параметрах генерации

Самый простой способ выбрать следующий токен — взять токен с максимальной вероятностью (`argmax` по логитам). Это называется **greedy decoding**. При фиксированном состоянии модели такой способ обычно детерминирован, но не позволяет исследовать альтернативные продолжения.

Для разнообразия применяют **sampling** — выбор токена из вероятностного распределения.

1. **`do_sample`** — разрешает случайное семплирование. При `do_sample=False` модель использует детерминированные способы декодирования (например, greedy при `num_beams=1`).
2. **`temperature`** — масштабирует логиты перед softmax (при семплировании):

![Формула softmax с температурой](assets/temperature_softmax.png)

- `T > 1`: распределение более равномерное.
- `0 < T < 1`: распределение более концентрированное.
- `T = 0`: в ряде inference API служит обозначением greedy, но в `transformers` для этого следует использовать `do_sample=False`.

![Как температура влияет на вероятности токенов](assets/temperature_distribution.png)

3. **`beam search`** — поддерживает `num_beams` лучших гипотез, расширяет их на каждом шаге и выбирает гипотезы с лучшими накопленными оценками. Параметры нормализации длины могут менять итоговый рейтинг.
4. **`top_k`** — оставляет для семплирования `k` наиболее вероятных токенов.
5. **`top_p`** (nucleus sampling) — оставляет минимальный набор токенов с суммарной вероятностью не меньше `p`.
6. **`max_new_tokens`** — максимальное число новых токенов ответа.
7. **`repetition_penalty`** и **`length_penalty`** — штраф за повторения и настройка оценки длины (зависит от метода декодирования).

**Практика:** для воспроизводимого жадного декодирования используем `do_sample=False`; для разнообразия — `do_sample=True` и настраиваем `temperature`, `top_k`, `top_p`. При сравнении экспериментов фиксируем параметры и, если применимо, seed.


In [ ]:
config = GenerationConfig.from_pretrained(MODEL_NAME)

_ = config.update(
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    top_k=20,
    max_new_tokens=256,
    num_beams=1,
)


In [ ]:
system_prompt = 'You are math assistant. Solve the provided math problem step by step and put your final answer within \\boxed{}'
prompt = 'Реши квадратное уравнение: x**2 + 1 = 0'

output = invoke(model_hf, tokenizer, prompt=prompt, system_prompt=system_prompt, config=config)

Документация: [`GenerationConfig` в Hugging Face Transformers](https://huggingface.co/docs/transformers/main_classes/text_generation). В разных бэкендах названия и допустимые значения параметров могут отличаться.


Поэкспериментируйте с параметрами генерации и входными промптами. Сравните ответы модели и сделайте выводы.


## Hugging Face Inference Providers API

[Документация Hugging Face Inference Providers](https://huggingface.co/docs/inference-providers/index). В этом варианте инференс выполняется у внешнего провайдера, а не на вашей GPU. Нужен Hugging Face API-токен; поддерживаемые провайдеры и модели могут меняться.


In [ ]:
from huggingface_hub import InferenceClient
from getpass import getpass

hf_api_key = os.environ.get("HF_TOKEN") or getpass("Hugging Face API token: ")
hf_client = InferenceClient(api_key=hf_api_key)

# Пример вызова chat completions через внешний inference provider.
# Доступность конкретной модели зависит от провайдера и вашего доступа.
hf_completion = hf_client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1",
    messages=[{"role": "user", "content": "Реши уравнение x**2 = 1. Ответь кратко."}],
    max_tokens=256,
)
print(hf_completion.choices[0].message.content)


## Внешний API-router: OpenRouter + LangChain

Роутер предоставляет доступ к моделям разных провайдеров по одному API. Для демонстрации используем OpenRouter с OpenAI-совместимым API и клиентом `ChatOpenAI` из `langchain-openai`. **Модель выполняется на стороне провайдера**, локальная GPU не требуется. Для вызова нужен API-ключ OpenRouter; запрос может быть платным.

Выберите доступную модель из [каталога OpenRouter](https://openrouter.ai/models). Пример ниже использует `openai/gpt-4o-mini`.

`ChatOpenAI` подходит для обычного OpenAI-совместимого Chat Completions API. Для специфичных возможностей OpenRouter (маршрутизация по провайдерам, специальные метаданные) существует отдельная интеграция `langchain-openrouter`, но здесь достаточно `ChatOpenAI`.


In [ ]:
from langchain_openai import ChatOpenAI
from getpass import getpass

model_name = "openai/gpt-4o-mini"
api_key = os.environ.get("OPENROUTER_API_KEY") or getpass("OpenRouter API key: ")

router_llm = ChatOpenAI(
    model=model_name,
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    use_responses_api=False,
)

response = router_llm.invoke("Реши уравнение x**2 = 1. Ответь кратко.")
print(response.content)


## vLLM

### Offline inference

При offline inference модель запускается через Python API `vllm.LLM` без HTTP-сервера. Например:

```python
from vllm import LLM, SamplingParams

llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct", max_model_len=2048)
result = llm.generate(["Реши уравнение x**2 = 1"], SamplingParams(temperature=0, max_tokens=128))
print(result[0].outputs[0].text)
```

Это **альтернатива** online serving, а не обязательный дополнительный запуск. Не загружайте одновременно модель через Hugging Face, offline vLLM и vLLM server на одну небольшую GPU — может не хватить VRAM. [Документация vLLM](https://docs.vllm.ai/en/stable/getting_started/quickstart/).


### Online inference

При **online inference** разворачивается OpenAI-совместимый сервер. Клиент обращается к `/v1/chat/completions` или `/v1/completions`, не загружая веса модели в свой процесс.

Для серверного запуска в этом семинаре используем `vllm serve`.


**Способы запуска:** выберите один из вариантов ниже — Docker, Colab или обычный терминал с GPU.

### Вариант 1. Docker (локальная машина или GPU-сервер)

Нужны Docker и NVIDIA Container Toolkit. Пример находится в [`run_vllm_docker_1.sh`](run_vllm_docker_1.sh). Из директории `lecture_2`:


```bash
bash run_vllm_docker_1.sh
```

Скрипт пробрасывает порт **11001** хоста на порт 8000 контейнера. Для такого запуска укажите клиенту `http://127.0.0.1:11001/v1`.


### Вариант 2. Google Colab (GPU)

1. Выберите **Runtime → Change runtime type → GPU** и убедитесь, что GPU доступна (`!nvidia-smi`).
2. В новой среде установите пакеты: `%pip install -U vllm openai requests`. После установки может потребоваться перезапуск runtime.
3. Если вы ранее загружали HF-модель в GPU-память в этом ноутбуке, **перезапустите runtime** перед vLLM, чтобы освободить VRAM.
4. Запустите следующую ячейку: сервер будет работать как отдельный процесс в фоне, логи — в `/tmp/vllm.log`. Для публичной модели токен HF не требуется.

Для бесплатных GPU (например, T4) выбран небольшой FP16-чекпойнт Qwen2.5-1.5B-Instruct, ограниченный контекст и умеренный лимит памяти. Доступность GPU и совместимость версии vLLM с окружением Colab зависят от runtime. Адрес `127.0.0.1` доступен из **того же runtime**, но не из вашего локального компьютера.


In [ ]:
import subprocess
import time
import requests
from pathlib import Path

VLLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
VLLM_PORT = 8000
VLLM_LOG = Path("/tmp/vllm.log")

# Запускаем сервер в отдельном процессе, чтобы ячейка не блокировалась.
log_handle = VLLM_LOG.open("w")
vllm_process = subprocess.Popen(
    [
        "vllm", "serve", VLLM_MODEL,
        "--host", "127.0.0.1",
        "--port", str(VLLM_PORT),
        "--dtype", "half",
        "--gpu-memory-utilization", "0.75",
        "--max-model-len", "2048",
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
log_handle.close()
print("vLLM PID:", vllm_process.pid)

# Дождёмся готовности API; первая загрузка может потребовать скачивания весов.
for _ in range(120):
    if vllm_process.poll() is not None:
        raise RuntimeError(f"vLLM завершился. Проверьте лог: {VLLM_LOG}")
    try:
        health = requests.get(f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=2)
        if health.ok:
            print("vLLM server is ready")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError(f"Сервер не готов. Посмотрите лог: {VLLM_LOG}")


### Вариант 3. Терминал на GPU-сервере

Вместо запуска из Python можно запустить сервер напрямую из терминала. Эта команда работает **в foreground**, поэтому для неё нужен отдельный терминал (или `tmux`). Не запускайте второй сервер поверх уже работающего на том же порту.


```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --host 127.0.0.1 --port 8000 \
  --dtype half --gpu-memory-utilization 0.75 --max-model-len 2048
```


Создаём OpenAI SDK-клиент. Он обращается к локальному серверу через `/v1/chat/completions`. Если сервер запущен через Docker-скрипт, поменяйте `base_url` на порт **11001**. Если задан `--api-key` у vLLM, передайте тот же ключ клиенту.


In [ ]:
from openai import OpenAI

vllm_client = OpenAI(
    api_key="EMPTY",  # произвольная строка, если vLLM запущен без --api-key
    base_url="http://127.0.0.1:8000/v1",  # для Docker: http://127.0.0.1:11001/v1
)


In [ ]:
# Обычный (не потоковый) запрос
completion = vllm_client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Реши уравнение: x**2 + 1 = 0."},
    ],
    temperature=0,
    max_tokens=256,
    stream=False,
)


Если `stream=False`, весь ответ возвращается после завершения генерации:


In [ ]:
print(completion.choices[0].message.content)


При `stream=True` сервер отправляет последовательность **чанков**, а не обязательно по одному токену на каждый чанк. Можно собрать текст ответа и приблизительно измерить TTFT как время до первого непустого текстового чанка:


In [ ]:
import time

start = time.perf_counter()
first_chunk_time = None
output_text = ""

stream = vllm_client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "Реши уравнение: x**2 + 1 = 0."}],
    temperature=0,
    max_tokens=256,
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content or ""
    if content:
        if first_chunk_time is None:
            first_chunk_time = time.perf_counter()
        output_text += content
        print(content, end="", flush=True)

total_time = time.perf_counter() - start
ttft = first_chunk_time - start if first_chunk_time is not None else None

print(f"\nTotal time: {total_time:.3f} s; TTFT (first text chunk): {ttft}")


Полезные ссылки:

1. [vLLM Quickstart](https://docs.vllm.ai/en/stable/getting_started/quickstart/)
2. [OpenAI-compatible server](https://docs.vllm.ai/en/stable/serving/openai_compatible_server/)
3. [vLLM CLI](https://docs.vllm.ai/en/stable/cli/)
4. [Performance tuning](https://docs.vllm.ai/en/stable/configuration/optimization/)


## Ollama (llama.cpp)

Пример локального запуска и сравнения квантизаций — в [домашнем задании](hw_1.ipynb).
